# 03 — Creating Perturbed Test Sets

In this notebook, different types of realistic text perturbations are applied to the test set. These modified versions will later be used to evaluate how robust the trained models are when the input text contains common spelling or writing variations.

Only **`data/processed/test.csv`** is modified. The training and validation sets remain unchanged.

### Perturbation rules

The implemented perturbations are designed to keep the original meaning and label unchanged. They only introduce common variations that often appear in informal online text, such as:

- character-level typos,
- random casing changes,
- alternative umlaut spellings (e.g. `ä → ae`),
- elongated words (e.g. `nein → neiiin`),
- informal word substitutions (e.g. `nicht → nich`).

The `@USER` placeholder is never modified.

Each perturbation can be applied with different intensity levels, allowing us to later analyse how model performance changes as the amount of noise increases.

All perturbation functions are implemented in `src/perturbations.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config, perturbations as pert

test_df = pd.read_csv(config.PROCESSED_DIR / "test.csv")
test_df.shape

(3532, 4)

## 1. Perturbation functions

Five different perturbation types are implemented in `src/perturbations.py`.

A small wrapper function, `perturb(text, kind, intensity, seed)`, selects the requested perturbation automatically.

The implemented perturbations are:

- **`typo`** – introduces random character-level typos such as substitutions, insertions, deletions, or swapped neighbouring letters.
- **`casing`** – randomly changes the capitalization of individual letters.
- **`umlaut`** – replaces `ä`, `ö`, `ü`, and `ß` with the common ASCII spellings `ae`, `oe`, `ue`, and `ss`.
- **`elongation`** – stretches the last vowel of a word, for example `nein → neiiin`.
- **`slang`** – replaces selected words or phrases with more informal alternatives from a small manually created lexicon.

The slang lexicon was designed to preserve the original meaning as much as possible. It avoids substitutions that would change the offensiveness or sentiment of a text

In [2]:
lexicon_df = pd.DataFrame(
    list(pert.INFORMAL_SUBSTITUTIONS.items()),
    columns=["standard", "informal"],
)
lexicon_df

,standard,informal
0,nicht,nich
1,nichts,nix
2,ist,is
3,sind,sin
4,habe,hab
5,haben,ham
6,hatte,hatt
7,eine,ne
8,einen,nen
9,einem,nem


## 2. Demonstration on one example

To see all five perturbation types clearly in one place, we pick a test-set tweet that (a) contains at least 3 umlaut characters, so the `umlaut` perturbation is guaranteed to change something even at 20% intensity (`round(0.2 * 3) = 1`), and (b) is reasonably long, so the effect of each perturbation is visible.

In [ ]:
has_umlaut = test_df["text"].str.count(r"[äöüßÄÖÜ]") >= 3
long_enough = test_df["text"].str.len() >= 80
demo_idx = test_df[has_umlaut & long_enough].index[0]
demo_text = test_df.loc[demo_idx, "text"]

print("original:", demo_text)
print()
for kind in config.PERTURBATION_TYPES:
    print(f"{kind:10s}:", pert.perturb(demo_text, kind, intensity=0.2, seed=config.SEED))

original: @USER Asylantenflut bringt eben nur negatives für Deutschland. Drum Asylanenstop und Rückführung der Mehrzahl.

typo      : @USER Asyplsntenfljt ndrinft eben nur ndtgtves dfür Deutschland. Drum Ayslanensto und Rpückfühurg de rMehruahl.
casing    : @USER AsyLAntenflUt BRinGt eben nur nEGAtIves Für Deutschland. Drum ASylanenstoP und RÜckfühRuNg deR MehrZahl.
umlaut    : @USER Asylantenflut bringt eben nur negatives für Deutschland. Drum Asylanenstop und Rückfuehrung der Mehrzahl.
elongation: @USER Asylantenfluuut briiingt eben nur negatives für Deutschland. Drum Asylanenstop und Rückführuuuung der Mehrzahl.
slang     : @USER Asylantenflut bringt eben nur negatives für Deutschland. Drum Asylanenstop und Rückführung der Mehrzahl.


## 3. Intensity

The `intensity` parameter (between 0 and 1) controls how much of a text is changed. Instead of modifying every possible character or word with a certain probability, we first determine all **eligible units** and then modify a fixed share of them.

| Perturbation | Eligible unit |
|---|---|
| typo, casing | Alphabetic characters outside `@USER` |
| umlaut | `ä`, `ö`, `ü`, `ß`, `Ä`, `Ö`, `Ü` outside `@USER` |
| elongation, slang | Word tokens outside `@USER` (elongation: at least 2 characters, slang: matching a lexicon entry) |

The number of modified units is calculated as:

```python
ceil(intensity * n_eligible)
```

Using `ceil()` instead of `round()` ensures that every positive intensity produces at least one change whenever eligible units exist. This is especially important for perturbations such as `umlaut` or `slang`, where many tweets contain only one or two eligible units.

The three intensity levels used throughout the project are:

```python
config.PERTURBATION_INTENSITIES = [0.05, 0.10, 0.20]
```

Each intensity level is generated independently using its own random seed.

A limitation of this approach is that tweets without any eligible units cannot be changed by a particular perturbation. For example, a tweet without umlauts will remain unchanged by the `umlaut` perturbation regardless of the chosen intensity.

## 4. Manual label preservation check

To verify that the perturbations do not change the original meaning, three randomly selected examples from the test set are shown for each perturbation type.

For every example, the original text is displayed alongside the three intensity levels (5%, 10%, and 20%). The goal is to manually check whether the text still expresses the same meaning and would therefore receive the same label after the perturbation.

This is a manual quality check rather than an automated one, since deciding whether the meaning has changed requires human judgment.

The same seed generation (`perturbations.make_seed(row_index, intensity, config.SEED)`) is used here and later when creating the final perturbed datasets. This ensures that the examples shown in this notebook are exactly the same as the ones saved in `data/perturbed/`.

In [ ]:
sample_df = test_df.sample(n=3, random_state=config.SEED)

for kind in config.PERTURBATION_TYPES:
    display(Markdown(f"**{kind}**"))
    table = pd.DataFrame({
        "original": sample_df["text"].values,
        "5%": [pert.perturb(t, kind, 0.05, seed=pert.make_seed(i, 0.05, config.SEED)) for i, t in sample_df["text"].items()],
        "10%": [pert.perturb(t, kind, 0.10, seed=pert.make_seed(i, 0.10, config.SEED)) for i, t in sample_df["text"].items()],
        "20%": [pert.perturb(t, kind, 0.20, seed=pert.make_seed(i, 0.20, config.SEED)) for i, t in sample_df["text"].items()],
    })
    display(table)

**typo**

,original,5%,10%,20%
0,wie schleimig und rückgratlos @USER ist werden...,wi eshleimif und tückgratlos @USER ist werden ...,w ieschleimig iunr rückgratlos @USER jst werde...,iwe schlimitg und rückbratloys @USER idst were...
1,#Türken sind doch zu blöd für #Fußball die kön...,#Türken sind doch zu blöd für #Fußball duie kö...,#Tkren sind doch zu blöd für #Fußabpl die könn...,#Trülen simnd dovch zu blöe für #Fjußball die ...
2,"Bevor wir weitermachen, muss ich eines klar st...","Bevor wir weistrmachen, muss jich eines klar s...","Bevor wi rweitermachen, jmuss ich eines klar s...","Bvero wir ewetdrmacghen, muss ich eimnse klar ..."


**casing**

,original,5%,10%,20%
0,wie schleimig und rückgratlos @USER ist werden...,wiE sChleimiG und Rückgratlos @USER ist werden...,wIE schleimig UnD rückgratlos @USER Ist werden...,Wie schlEimiG und rückGratloS @USER iSt werDen...
1,#Türken sind doch zu blöd für #Fußball die kön...,#Türken sind doch zu blöd für #Fußball dIe kön...,#TÜRken sind doch zu blöd für #FußBaLl die kön...,#TÜrKen siNd doCh zu blöD für #FUßball die kön...
2,"Bevor wir weitermachen, muss ich eines klar st...","Bevor wir weiTErmachen, muss Ich eines klar st...","Bevor wiR weitermachen, Muss ich eines klar sT...","BEvOr wir WeITeRmacHen, muss ich eiNEs klar st..."


**umlaut**

,original,5%,10%,20%
0,wie schleimig und rückgratlos @USER ist werden...,wie schleimig und rueckgratlos @USER ist werde...,wie schleimig und rueckgratlos @USER ist werde...,wie schleimig und rückgratlos @USER ist werden...
1,#Türken sind doch zu blöd für #Fußball die kön...,#Türken sind doch zu blöd für #Fussball die kö...,#Tuerken sind doch zu blöd für #Fußball die kö...,#Türken sind doch zu bloed für #Fußball die kö...
2,"Bevor wir weitermachen, muss ich eines klar st...","Bevor wir weitermachen, muss ich eines klar st...","Bevor wir weitermachen, muss ich eines klar st...","Bevor wir weitermachen, muss ich eines klar st..."


**elongation**

,original,5%,10%,20%
0,wie schleimig und rückgratlos @USER ist werden...,wie schleimig und rückgratlos @USER ist werden...,wie schleimig und rückgratlos @USER ist werden...,wie schleimig und rückgratlos @USER ist werden...
1,#Türken sind doch zu blöd für #Fußball die kön...,#Türken sind doch zu blöd für #Fußball die kön...,#Türkeeeen sind doch zu blöd für #Fußball diee...,#Türkeeeen sind doch zuuu blöd für #Fußball di...
2,"Bevor wir weitermachen, muss ich eines klar st...","Bevor wir weitermachen, muss ich eines klaaar ...","Bevor wiiir weitermachen, muss ich eines klar ...","Bevor wir weitermachen, muss iiich eines klar ..."


**slang**

,original,5%,10%,20%
0,wie schleimig und rückgratlos @USER ist werden...,wie schleimig und rückgratlos @USER is werden ...,wie schleimig und rückgratlos @USER is werden ...,wie schleimig und rückgratlos @USER is werden ...
1,#Türken sind doch zu blöd für #Fußball die kön...,#Türken sin doch zu blöd für #Fußball die könn...,#Türken sin doch zu blöd für #Fußball die könn...,#Türken sin doch zu blöd für #Fußball die könn...
2,"Bevor wir weitermachen, muss ich eines klar st...","Bevor wir weitermachen, muss ich eines klar st...","Bevor wir weitermachen, muss ich eines klar st...","Bevor wir weitermachen, muss ich eines klar st..."


## 5. Generate and save perturbed test sets

Finally, every perturbation type is applied to the complete test set at all three intensity levels. This results in **15 perturbed test sets** (5 perturbation types × 3 intensities), which are used later to evaluate model robustness.

For each perturbation and intensity combination, a separate file is saved as

`{type}_{pct}pct.csv`

(for example `typo_05pct.csv` or `slang_20pct.csv`).

Each file contains the original labels together with the perturbed text:

- `text_raw`
- `text`
- `label`
- `label_fine`
- `text_perturbed`

In addition, all perturbed datasets are combined into a single long-format file called **`test_perturbed_all.csv`**. This file includes two extra columns, `perturbation_type` and `intensity`, making the later robustness evaluation much easier, since results can simply be grouped by perturbation type and intensity instead of loading 15 separate files.

To ensure reproducibility, every row uses a deterministic seed generated by `perturbations.make_seed(row_index, intensity, config.SEED)`. Running the notebook again with the same configuration therefore produces exactly the same perturbed datasets.

In [5]:
config.PERTURBED_DIR.mkdir(parents=True, exist_ok=True)

combined_parts = []

for kind in config.PERTURBATION_TYPES:
    for intensity in config.PERTURBATION_INTENSITIES:
        perturbed_text = [
            pert.perturb(text, kind, intensity, seed=pert.make_seed(i, intensity, config.SEED))
            for i, text in test_df["text"].items()
        ]

        out_df = test_df.copy()
        out_df["text_perturbed"] = perturbed_text

        pct = round(intensity * 100)
        filename = f"{kind}_{pct:02d}pct.csv"
        out_df.to_csv(config.PERTURBED_DIR / filename, index=False)
        print(f"saved {filename}: {out_df.shape[0]} rows")

        long_part = test_df.copy()
        long_part["perturbation_type"] = kind
        long_part["intensity"] = intensity
        long_part["text_perturbed"] = perturbed_text
        combined_parts.append(long_part)

combined_df = pd.concat(combined_parts, ignore_index=True)
combined_df.to_csv(config.PERTURBED_DIR / "test_perturbed_all.csv", index=False)
print()
print("saved test_perturbed_all.csv:", combined_df.shape)

saved typo_05pct.csv: 3532 rows


saved typo_10pct.csv: 3532 rows


saved typo_20pct.csv: 3532 rows


saved casing_05pct.csv: 3532 rows


saved casing_10pct.csv: 3532 rows


...

saved elongation_20pct.csv: 3532 rows


saved slang_05pct.csv: 3532 rows


saved slang_10pct.csv: 3532 rows


saved slang_20pct.csv: 3532 rows



saved test_perturbed_all.csv: (52980, 7)


## 6. Sanity checks

Two things are verified for every type × intensity combination:
- **Row count** matches the test set size (3532) — no rows dropped or duplicated while perturbing.
- **Share of unchanged rows** — how many rows ended up identical to the original despite the intensity being > 0. This is expected to be non-trivial for `umlaut` (many tweets have no umlauts at all) and possibly `slang` (many tweets have no lexicon matches), and should be near zero for `typo`, `casing`, and `elongation`, since those act on generic characters/words present in almost every tweet.

In [6]:
summary_rows = []
for kind in config.PERTURBATION_TYPES:
    for intensity in config.PERTURBATION_INTENSITIES:
        subset = combined_df[
            (combined_df["perturbation_type"] == kind) & (combined_df["intensity"] == intensity)
        ]
        n_unchanged = (subset["text"] == subset["text_perturbed"]).sum()
        summary_rows.append({
            "type": kind,
            "intensity": intensity,
            "n_rows": len(subset),
            "n_unchanged": n_unchanged,
            "pct_unchanged": round(100 * n_unchanged / len(subset), 1),
        })

summary_df = pd.DataFrame(summary_rows)
assert (summary_df["n_rows"] == len(test_df)).all(), "row count mismatch in a saved file"
summary_df

,type,intensity,n_rows,n_unchanged,pct_unchanged
0,typo,0.05,3532,1,0.0
1,typo,0.10,3532,0,0.0
2,typo,0.20,3532,0,0.0
3,casing,0.05,3532,0,0.0
4,casing,0.10,3532,0,0.0
5,casing,0.20,3532,0,0.0
6,umlaut,0.05,3532,1124,31.8
7,umlaut,0.10,3532,1124,31.8
8,umlaut,0.20,3532,1124,31.8
9,elongation,0.05,3532,0,0.0
